In [1]:
!pip install transformers datasets torch seqeval scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.5 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=6d7c9c7e570a05fe75f131a7bc5d2b1cd51d14ffaec0d1d1b5ddae4b12d3bb0d
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency 

In [2]:
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall
import json
import torch
import numpy as np
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall
from sklearn.metrics import f1_score as sklearn_f1, precision_score as sklearn_precision, recall_score as sklearn_recall, classification_report
import random
from sklearn.model_selection import train_test_split
import warnings
from sklearn.metrics import f1_score as sklearn_f1, precision_score as sklearn_precision, recall_score as sklearn_recall

In [3]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
# نرمال‌سازی متن برای مدیریت نیم‌فاصله‌ها
def normalize_text(text):

    text = text.replace('‌', ' <ZWNJ> ')
    return text.strip()

def create_tokens_and_labels(sample):
    utt = normalize_text(sample['utt'])
    tokens = utt.split()
    labels = ['O'] * len(tokens)

    annot = normalize_text(sample['annot_utt']).split()
    current_label = 'O'
    token_idx = 0

    for word in annot:
        if word.startswith('['):
            current_label = word.strip('[]')
        elif word == ']':
            current_label = 'O'
        elif current_label != 'O' and token_idx < len(labels):
            if labels[token_idx] == f"B-{current_label}":
                labels[token_idx] = f"I-{current_label}"
            else:
                labels[token_idx] = f"B-{current_label}"
            token_idx += 1
        elif token_idx < len(labels):
            token_idx += 1

    tokens = [token.replace('<ZWNJ>', '‌') for token in tokens]
    return tokens, labels, sample['intent']


data_path = '/content/drive/MyDrive/fa-IR.jsonl'
with open(data_path, 'r', encoding='utf-8') as f:
    massive_raw = [json.loads(line) for line in f]

sentences_tr, tags_tr, intents_tr = [], [], []
sentences_val, tags_val, intents_val = [], [], []
sentences_test, tags_test, intents_test = [], [], []
for sample in massive_raw:
    tokens, labels, intent = create_tokens_and_labels(sample)
    if sample['partition'] == 'train':
        sentences_tr.append(tokens)
        tags_tr.append(labels)
        intents_tr.append(intent)
    elif sample['partition'] == 'dev':
        sentences_val.append(tokens)
        tags_val.append(labels)
        intents_val.append(intent)
    elif sample['partition'] == 'test':
        sentences_test.append(tokens)
        tags_test.append(labels)
        intents_test.append(intent)

sentences_tr, _, tags_tr, _, intents_tr, _ = train_test_split(
    sentences_tr, tags_tr, intents_tr, test_size=0.9, random_state=42)


if len(sentences_val) >= 200:
    val_indices = random.sample(range(len(sentences_val)), 200)
    sentences_val = [sentences_val[i] for i in val_indices]
    tags_val = [tags_val[i] for i in val_indices]
    intents_val = [intents_val[i] for i in val_indices]

if len(sentences_test) >= 200:
    test_indices = random.sample(range(len(sentences_test)), 200)
    sentences_test = [sentences_test[i] for i in test_indices]
    tags_test = [tags_test[i] for i in test_indices]
    intents_test = [intents_test[i] for i in test_indices]


slot_labels = sorted(set(tag for tags in tags_tr + tags_val + tags_test for tag in tags))
intent_labels = sorted(set(intents_tr + intents_val + intents_test))
label_list = slot_labels + intent_labels


def create_prompt(sentence, intent, slots):
    return f"Input: {' '.join(sentence)}\nOutput Intent: {intent}\nOutput Slots: {' '.join(slots)}"

train_data = [create_prompt(s, i, t) for s, i, t in zip(sentences_tr, intents_tr, tags_tr)]
val_data = [create_prompt(s, i, t) for s, i, t in zip(sentences_val, intents_val, tags_val)]
test_data = [create_prompt(s, i, t) for s, i, t in zip(sentences_test, intents_test, tags_test)]


train_dataset = Dataset.from_dict({'text': train_data, 'slot_labels': tags_tr})
val_dataset = Dataset.from_dict({'text': val_data, 'slot_labels': tags_val})
test_dataset = Dataset.from_dict({'text': test_data, 'slot_labels': tags_test})


tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=64,
        return_tensors='pt'
    )
    labels = []
    for slot_labels in examples['slot_labels']:
        label_ids = [-100] * len(tokenized['input_ids'][0])
        for idx, slot in enumerate(slot_labels):
            if idx < len(label_ids):
                label_ids[idx] = slot_labels.index(slot)
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

tokenized_datasets = DatasetDict({
    'train': train_dataset.map(tokenize_function, batched=True),
    'validation': val_dataset.map(tokenize_function, batched=True),
    'test': test_dataset.map(tokenize_function, batched=True),
})


training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    logging_dir='./logs',
    load_best_model_at_end=True,
    save_total_limit=2,
    report_to="none",
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=10,
    save_steps=10,
    save_strategy="steps",
    fp16=True,
    dataloader_num_workers=4,
    log_level="info",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
)


trainer.train()
trainer.save_model('./trained_gpt2_model')

print("Training and saving completed.")



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Using auto half precision backend
The following columns in the training set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 1,151
  Num Epochs = 5
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 8
  Total optimization steps = 180
  Number of trainable parameters = 124,440,576


Step,Training Loss,Validation Loss
10,10.539400,4.949884
20,2.746400,1.923445
30,2.003000,1.799048
40,1.923400,1.711872
50,1.748000,1.569236
60,1.676400,1.603160
70,1.647400,1.540890
80,1.660400,1.523412
90,1.598500,1.522228
100,1.621100,1.532289


The following columns in the evaluation set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 200
  Batch size = 4
Saving model checkpoint to ./results/checkpoint-10
Configuration saved in ./results/checkpoint-10/config.json
Configuration saved in ./results/checkpoint-10/generation_config.json
Model weights saved in ./results/checkpoint-10/model.safetensors
tokenizer config file saved in ./results/checkpoint-10/tokenizer_config.json
Special tokens file saved in ./results/checkpoint-10/special_tokens_map.json
The following columns in the evaluation set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Eval

Training and saving completed.


In [8]:
def get_labels(predictions, label_ids):
    preds = np.argmax(predictions, axis=-1)
    labels = label_ids

    preds = preds.tolist()
    labels = labels.tolist()

    true_labels = []
    true_preds = []

    for pred, label in zip(preds, labels):
        temp_true_labels = []
        temp_true_preds = []
        for p, l in zip(pred, label):
            if l != -100:
                temp_true_labels.append(l)
                temp_true_preds.append(p)
        true_labels.append(temp_true_labels)
        true_preds.append(temp_true_preds)

    return true_labels, true_preds

def id_to_label(id_list, label_map):
    return [[label_map[id] for id in labels] for labels in id_list]


def predict_in_batches(trainer, dataset, batch_size=16):
    predictions = []
    label_ids = []
    for i in range(0, len(dataset), batch_size):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))
        results = trainer.predict(batch)
        predictions.append(results.predictions)
        label_ids.append(results.label_ids)
        torch.cuda.empty_cache()
    predictions = np.concatenate(predictions, axis=0)
    label_ids = np.concatenate(label_ids, axis=0)
    return predictions, label_ids

val_predictions, val_label_ids = predict_in_batches(trainer, tokenized_datasets['validation'], batch_size=16)
val_true_labels, val_preds = get_labels(val_predictions, val_label_ids)
val_true_labels_text = id_to_label(val_true_labels, label_list)
val_preds_text = id_to_label(val_preds, label_list)


test_predictions, test_label_ids = predict_in_batches(trainer, tokenized_datasets['test'], batch_size=16)
test_true_labels, test_preds = get_labels(test_predictions, test_label_ids)
test_true_labels_text = id_to_label(test_true_labels, label_list)
test_preds_text = id_to_label(test_preds, label_list)


print("\nValidation Slot Filling Classification Report (seqeval):")
print(seqeval_classification_report(val_true_labels_text, val_preds_text, digits=4))


print("\nTest Slot Filling Classification Report (seqeval):")
print(seqeval_classification_report(test_true_labels_text, test_preds_text, digits=4))


val_f1_micro = f1_score(val_true_labels_text, val_preds_text, average='micro')
val_precision_micro = precision_score(val_true_labels_text, val_preds_text, average='micro')
val_recall_micro = recall_score(val_true_labels_text, val_preds_text, average='micro')

val_f1_macro = f1_score(val_true_labels_text, val_preds_text, average='macro')
val_precision_macro = precision_score(val_true_labels_text, val_preds_text, average='macro')
val_recall_macro = recall_score(val_true_labels_text, val_preds_text, average='macro')

print("\nValidation Slot Filling Metrics (Micro):")
print(f"Precision: {val_precision_micro:.4f}")
print(f"Recall: {val_recall_micro:.4f}")
print(f"F1 Score: {val_f1_micro:.4f}")

print("\nValidation Slot Filling Metrics (Macro):")
print(f"Precision: {val_precision_macro:.4f}")
print(f"Recall: {val_recall_macro:.4f}")
print(f"F1 Score: {val_f1_macro:.4f}")

test_f1_micro = f1_score(test_true_labels_text, test_preds_text, average='micro')
test_precision_micro = precision_score(test_true_labels_text, test_preds_text, average='micro')
test_recall_micro = recall_score(test_true_labels_text, test_preds_text, average='micro')

test_f1_macro = f1_score(test_true_labels_text, test_preds_text, average='macro')
test_precision_macro = precision_score(test_true_labels_text, test_preds_text, average='macro')
test_recall_macro = recall_score(test_true_labels_text, test_preds_text, average='macro')

print("\nTest Slot Filling Metrics (Micro):")
print(f"Precision: {test_precision_micro:.4f}")
print(f"Recall: {test_recall_micro:.4f}")
print(f"F1 Score: {test_f1_micro:.4f}")

print("\nTest Slot Filling Metrics (Macro):")
print(f"Precision: {test_precision_macro:.4f}")
print(f"Recall: {test_recall_macro:.4f}")
print(f"F1 Score: {test_f1_macro:.4f}")


val_true_intents = [labels[-1] for labels in val_true_labels if len(labels) > 0]
val_pred_intents = [preds[-1] for preds in val_preds if len(preds) > 0]
val_true_intents_text = [label_list[id] for id in val_true_intents]
val_pred_intents_text = [label_list[id] for id in val_pred_intents]

print("\nValidation Intent Classification Report (sklearn):")
print(sklearn_classification_report(val_true_intents_text, val_pred_intents_text, digits=4))

val_intent_f1_micro = sklearn_f1(val_true_intents_text, val_pred_intents_text, average='micro')
val_intent_precision_micro = sklearn_precision(val_true_intents_text, val_pred_intents_text, average='micro')
val_intent_recall_micro = sklearn_recall(val_true_intents_text, val_pred_intents_text, average='micro')

val_intent_f1_macro = sklearn_f1(val_true_intents_text, val_pred_intents_text, average='macro')
val_intent_precision_macro = sklearn_precision(val_true_intents_text, val_pred_intents_text, average='macro')
val_intent_recall_macro = sklearn_recall(val_true_intents_text, val_pred_intents_text, average='macro')

print("\nValidation Intent Metrics (Micro):")
print(f"Precision: {val_intent_precision_micro:.4f}")
print(f"Recall: {val_intent_recall_micro:.4f}")
print(f"F1 Score: {val_intent_f1_micro:.4f}")

print("\nValidation Intent Metrics (Macro):")
print(f"Precision: {val_intent_precision_macro:.4f}")
print(f"Recall: {val_intent_recall_macro:.4f}")
print(f"F1 Score: {val_intent_f1_macro:.4f}")

test_true_intents = [labels[-1] for labels in test_true_labels if len(labels) > 0]
test_pred_intents = [preds[-1] for preds in test_preds if len(preds) > 0]
test_true_intents_text = [label_list[id] for id in test_true_intents]
test_pred_intents_text = [label_list[id] for id in test_pred_intents]

print("\nTest Intent Classification Report (sklearn):")
print(sklearn_classification_report(test_true_intents_text, test_pred_intents_text, digits=4))

test_intent_f1_micro = sklearn_f1(test_true_intents_text, test_pred_intents_text, average='micro')
test_intent_precision_micro = sklearn_precision(test_true_intents_text, test_pred_intents_text, average='micro')
test_intent_recall_micro = sklearn_recall(test_true_intents_text, test_pred_intents_text, average='micro')

test_intent_f1_macro = sklearn_f1(test_true_intents_text, test_pred_intents_text, average='macro')
test_intent_precision_macro = sklearn_precision(test_true_intents_text, test_pred_intents_text, average='macro')
test_intent_recall_macro = sklearn_recall(test_true_intents_text, test_pred_intents_text, average='macro')

print("\nTest Intent Metrics (Micro):")
print(f"Precision: {test_intent_precision_micro:.4f}")
print(f"Recall: {test_intent_recall_micro:.4f}")
print(f"F1 Score: {test_intent_f1_micro:.4f}")

print("\nTest Intent Metrics (Macro):")
print(f"Precision: {test_intent_precision_macro:.4f}")
print(f"Recall: {test_intent_recall_macro:.4f}")
print(f"F1 Score: {test_intent_f1_macro:.4f}")


The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 16
  Batch size = 4


The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 16
  Batch size = 4


The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 16
  Batch size = 4


The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 16
  Batch size = 4


The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 16
  Batch size = 4


...

The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 16
  Batch size = 4


The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 16
  Batch size = 4


The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 16
  Batch size = 4


The following columns in the test set don't have a corresponding argument in `GPT2LMHeadModel.forward` and have been ignored: slot_labels, text. If slot_labels, text are not expected by `GPT2LMHeadModel.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 8
  Batch size = 4



Validation Slot Filling Classification Report (seqeval):
                   precision    recall  f1-score   support

       alarm_type     0.5728    0.9542    0.7158       895
         app_name     0.0000    0.0000    0.0000       152
      artist_name     0.0000    0.0000    0.0000        81
 audiobook_author     0.0000    0.0000    0.0000       104
   audiobook_name     0.1429    0.0100    0.0187       100
    business_name     0.0563    0.0430    0.0488        93
    business_type     0.0000    0.0000    0.0000        54
    change_amount     0.1000    0.0213    0.0351        47
       color_type     0.0000    0.0000    0.0000        30
     cooking_type     0.0435    0.0526    0.0476        19
    currency_name     0.0000    0.0000    0.0000         4
             date     0.0000    0.0000    0.0000        10
  definition_word     0.0000    0.0000    0.0000        12
      device_type     0.0000    0.0000    0.0000         3
       drink_type     0.0000    0.0000    0.0000        